# QM9 Comparison — Physics-Informed Compressor vs PCA

This notebook is ready to upload to **Google Colab**.

**Setup in Colab:** run the first code cell and upload two files when prompted:

1. `molcompress-0.1.0-py3-none-any.whl` (the installable algorithm package, in `ISEF/dist/`)
2. `qm9.csv` (optional — if omitted, the notebook either looks on Google Drive at
   `/content/drive/MyDrive/qm9.csv` or falls back to labelled synthetic demo data)

The experiment trains both compressors on the same train split and scaler, then
compares reconstruction MSE and the physical law `|gap − (lumo − homo)|` on the
held-out test set.

> Caveat: `gap = lumo − homo` is linear and PCA is a linear projector, so PCA
> preserves this identity by construction. The PINN's real advantage is
> reconstruction fidelity.

In [1]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
CSV_PATH = None

if IN_COLAB:
    from google.colab import files
    print('Upload the molcompress wheel and (optionally) qm9.csv:')
    uploaded = files.upload()
    base = Path('/content')
    wheels = [base / n for n in uploaded if n.lower().endswith('.whl')]
    csvs = [base / n for n in uploaded if n.lower().endswith('.csv')]

    if wheels:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', str(wheels[0])])
        print('Installed', wheels[0].name)
    else:
        print('No wheel uploaded; trying pip install molcompress from PyPI ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'molcompress'])

    if csvs:
        CSV_PATH = csvs[0]
    else:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            drive_csv = Path('/content/drive/MyDrive/qm9.csv')
            CSV_PATH = drive_csv if drive_csv.is_file() else None
        except Exception as exc:
            print('Drive mount skipped:', exc)
else:
    local_csv = Path('/home/joseph/Downloads/qm9.csv')
    CSV_PATH = local_csv if local_csv.is_file() else None

print('IN_COLAB:', IN_COLAB)
print('CSV_PATH:', CSV_PATH)

IN_COLAB: False
CSV_PATH: /home/joseph/Downloads/qm9.csv


In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from molcompress import (
    Compressor, compare, demo_data, features, load_dataset, save_artifacts,
)

if CSV_PATH is not None:
    df, label = load_dataset(CSV_PATH, seed=42, max_samples=0, clip_percentile=99.9)
    X = features(df)
else:
    df = demo_data(4000, seed=42)
    X = features(df)
    label = 'SYNTHETIC DEMO DATA (no QM9 CSV available)'

print(label)
print('Feature matrix:', X.shape)
display(df.head())

QM9 CSV: /home/joseph/Downloads/qm9.csv (n=133885) -- converted homo/lumo/gap from Hartree to eV; clipped 134 A outliers at 17.858; clipped 134 B outliers at 4.369; clipped 134 C outliers at 3.446
Feature matrix: (133885, 11)


,A,B,C,alpha,homo,lumo,gap,r2,zpve,cv,u0
0,17.857805,4.369451,3.445679,13.21,-10.549854,3.186453,13.736308,35.3641,0.044749,6.469,-40.478930
1,17.857805,4.369451,3.445679,9.46,-6.993326,2.255824,9.249150,26.1563,0.034358,6.316,-56.525887
2,17.857805,4.369451,3.445679,6.31,-7.967494,1.869422,9.836916,19.0002,0.021375,6.002,-76.404702
3,0.000000,4.369451,3.445679,16.28,-7.741639,1.376896,9.118536,59.5248,0.026841,8.574,-77.308427
4,0.000000,4.369451,3.445679,12.99,-9.806984,0.519737,10.329442,48.7476,0.016601,6.278,-93.411888


## Train and compare

`molcompress.compare()` does the whole experiment:

- 80% train / 20% test (seed 42)
- `Compressor` (3D physics-informed autoencoder) trained with physics weight 5.0
- 3-component PCA fitted on the same standardised training matrix
- both reconstruct the same test set; metrics are computed with one shared function

In [3]:
result = compare(
    X,
    test_size=0.2,
    latent_dim=3,
    physics_weight=5.0,
    epochs=80,
    batch_size=512,
    seed=42,
    verbose=False,
)
metrics = result['metrics']
print('epochs run:', len(result['model'].history_['train_loss']))
print('acceptance threshold:', round(result['threshold'], 5), 'eV')
print('PCA explained variance:', np.round(result['pca'].explained_variance_ratio_, 4),
      'total:', round(float(result['pca'].explained_variance_ratio_.sum()), 4))
display(metrics.round(6))

epochs run: 80
acceptance threshold: 0.05 eV
PCA explained variance: [0.3769 0.2577 0.1262] total: 0.7608


,method,recon_mse_standardized,recon_mse_physical,electronics_mse_ev2,violation_mean_ev,violation_std_ev,violation_p95_ev,violation_max_ev
0,Original (uncompressed),0.000000,0.000000,0.000000,0.000686,0.001182,0.002721,0.002721
1,PCA (3 components),0.238925,706.239403,0.273920,0.000008,0.000006,0.000018,0.000050
2,Physics-Informed AE (3D),0.087427,177.457875,0.104282,0.004903,0.004497,0.012907,0.103175


In [4]:
pca_m = result['metrics_pca']
pinn_m = result['metrics_pinn']
mse_drop = 100 * (pca_m['recon_mse_standardized'] - pinn_m['recon_mse_standardized']) / pca_m['recon_mse_standardized']
elec_drop = 100 * (pca_m['electronics_mse_ev2'] - pinn_m['electronics_mse_ev2']) / pca_m['electronics_mse_ev2']

print(f'PINN reconstruction MSE reduction vs PCA: {mse_drop:.1f}%')
print(f'PINN electronic-feature MSE reduction vs PCA: {elec_drop:.1f}%')
print(f'Mean violation — PCA: {pca_m["violation_mean_ev"]:.8f} eV | '
      f'PINN: {pinn_m["violation_mean_ev"]:.8f} eV')

PINN reconstruction MSE reduction vs PCA: 63.4%
PINN electronic-feature MSE reduction vs PCA: 61.9%
Mean violation — PCA: 0.00000763 eV | PINN: 0.00490335 eV


In [5]:
bar = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Reconstruction MSE (standardised, lower is better)',
                    'Physical violation |gap - (lumo - homo)| (eV, lower is better)'),
)
colors = {'PCA (3 components)': '#636efa', 'Physics-Informed AE (3D)': '#2ca02c'}
for _, row in metrics.iterrows():
    if row['method'] == 'Original (uncompressed)':
        continue
    bar.add_trace(go.Bar(
        x=[row['method']], y=[row['recon_mse_standardized']], name=row['method'],
        marker_color=colors[row['method']],
        text=[f"{row['recon_mse_standardized']:.4f}"], textposition='outside'),
        row=1, col=1)
    bar.add_trace(go.Bar(
        x=[row['method']], y=[row['violation_mean_ev']], name=row['method'],
        marker_color=colors[row['method']], showlegend=False,
        text=[f"{row['violation_mean_ev']:.6f} eV"], textposition='outside'),
        row=1, col=2)
bar.update_layout(height=430, barmode='group', title='QM9: PINN vs PCA (test set)')
display(bar)

In [6]:
emb = result['embeddings']
plot_n = min(3000, len(emb))
idx = np.random.default_rng(42).choice(len(emb), size=plot_n, replace=False)
all_v = np.concatenate([emb['violation_pca'].to_numpy()[idx], emb['violation_pinn'].to_numpy()[idx]])
cmin, cmax = float(all_v.min()), float(max(all_v.max(), all_v.min() + 1e-9))

def latent_figure(prefix, title, axes):
    fig = go.Figure(go.Scatter3d(
        x=emb[f'{prefix}_1'].to_numpy()[idx],
        y=emb[f'{prefix}_2'].to_numpy()[idx],
        z=emb[f'{prefix}_3'].to_numpy()[idx],
        mode='markers',
        marker=dict(size=3, color=emb[f'violation_{prefix}'].to_numpy()[idx],
                    colorscale='RdYlGn_r', cmin=cmin, cmax=cmax, opacity=0.8,
                    colorbar=dict(title='Violation (eV)', thickness=12)),
        hovertemplate='<b>Sample %{text}</b><br>' + f'{axes[0]}: %{{x:.3f}}<br>'
                      + f'{axes[1]}: %{{y:.3f}}<br>{axes[2]}: %{{z:.3f}}<extra></extra>',
        text=emb['sample_id'].to_numpy()[idx],
    ))
    fig.update_layout(title=title, height=520, margin=dict(l=0, r=0, b=0, t=45),
                      scene=dict(xaxis_title=axes[0], yaxis_title=axes[1],
                                 zaxis_title=axes[2], aspectmode='cube'))
    return fig

display(latent_figure('pca', 'PCA latent space (3 components)', ('PC1', 'PC2', 'PC3')))
display(latent_figure('pinn', 'Physics-Informed AE latent space (3D)', ('z1', 'z2', 'z3')))

In [7]:
out_dir = Path('/content/artifacts') if IN_COLAB else Path.cwd() / 'artifacts'
saved = save_artifacts(result, out_dir, label)
print('Saved model, PCA, metrics and embeddings to:', saved.resolve())

Saved model, PCA, metrics and embeddings to: /home/joseph/Work/ISEF/artifacts


## Interpretation

- The physics-informed autoencoder reconstructs the QM9 descriptors much better than 3-component PCA (lower MSE overall and on the electronic features).
- PCA's violation score is near zero because a linear projector preserves the linear identity `gap = lumo − homo`. This is mathematical, not learned physics.
- Both mean violations are far below the calibrated 0.05 eV threshold; the screener's raw-input check is what catches genuinely inconsistent molecules.
- To rerun with a different QM9 release, upload that CSV in the setup cell.